# MironLaw 1.1 — Colab QLoRA Training
**Model:** Qwen2.5-7B-Instruct + LoRA → push to `mironintelligence/MironLaw-1.1`

**Gereksinim:** Runtime > Change runtime type > **T4 GPU** seç

**Token:** Sol panel > 🔑 Secrets > `HF_TOKEN` ekle

In [ ]:
%%capture
!pip install -q transformers datasets huggingface_hub peft accelerate sentencepiece
!pip install -q bitsandbytes
!pip install -q "trl>=0.12.0,<0.14.0"

In [ ]:
import os, json, torch, time, gc
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    default_data_collator, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

# HF Token — Colab Secrets'tan al (sol panel > kiliт ikonu > HF_TOKEN ekle)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if not HF_TOKEN:
        raise ValueError('empty')
    print('Token Colab Secrets\'tan alindi.')
except Exception:
    import getpass
    HF_TOKEN = getpass.getpass('HuggingFace token gir (hf_...): ')

login(token=HF_TOKEN)

# GPU kontrolu
if not torch.cuda.is_available():
    raise RuntimeError('GPU yok! Runtime > Change runtime type > T4 GPU sec!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
print(f'PyTorch: {torch.__version__}')

MODEL_NAME   = 'Qwen/Qwen2.5-7B-Instruct'
HF_REPO      = 'mironintelligence/MironLaw-1.1'
ADAPTER_DIR  = '/tmp/mironlaw_adapter'
MAX_LEN      = 512
NUM_SAMPLES  = 100_000
MAX_STEPS    = 780

# ── QLoRA config ──────────────────────────────────────────────────────────────
# 4-bit NF4: 7B model ~3.5 GB VRAM (vs 14 GB FP16) — tek T4'e sigdirir
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Tokenizer yukleniyor...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

print('Model yukleniyor (QLoRA 4-bit, FP16 compute)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto', token=HF_TOKEN,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
))
model.print_trainable_parameters()

SYSTEM = 'Sen MironLaw, Turk hukuku konusunda uzmanlasmis bir AI asistanisin.'

def fmt(ex):
    msgs = ex.get('messages', '')
    if isinstance(msgs, str):
        try: msgs = json.loads(msgs)
        except: msgs = []
    if not msgs:
        msgs = [
            {'role': 'system',    'content': ex.get('system', SYSTEM)},
            {'role': 'user',      'content': ex.get('instruction', ex.get('user', ''))},
            {'role': 'assistant', 'content': ex.get('output', ex.get('assistant', ''))},
        ]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

print(f'Dataset yukleniyor ({NUM_SAMPLES:,} ornek)...')
raw = load_dataset('mironbaba/mironlaw-train-data', split='train')
print(f'Toplam dataset: {len(raw):,}')
raw = raw.shuffle(seed=42).select(range(min(NUM_SAMPLES, len(raw))))
ds  = raw.map(fmt, remove_columns=raw.column_names, num_proc=2)
ds  = ds.filter(lambda x: len(x['text']) > 80)
print(f'Format sonrasi: {len(ds):,}')

print(f'Tokenize ediliyor (max_length={MAX_LEN})...')
t0 = time.time()

def tokenize_fn(batch):
    enc = tokenizer(
        batch['text'], truncation=True, max_length=MAX_LEN,
        padding='max_length', return_attention_mask=True,
    )
    enc['labels'] = [
        [-100 if m == 0 else i for i, m in zip(ids, masks)]
        for ids, masks in zip(enc['input_ids'], enc['attention_mask'])
    ]
    return enc

ds_tok = ds.map(tokenize_fn, batched=True, batch_size=256,
                remove_columns=['text'], num_proc=2)
print(f'Tokenizasyon tamamlandi: {time.time()-t0:.1f}s')

# ── Egitim ────────────────────────────────────────────────────────────────────
print(f'Egitim basliyor (QLoRA, FP16, {MAX_STEPS} adim)...')
trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=ds_tok, data_collator=default_data_collator,
    args=SFTConfig(
        max_seq_length=MAX_LEN, packing=False,
        per_device_train_batch_size=4, gradient_accumulation_steps=16,
        gradient_checkpointing=True, warmup_steps=20,
        max_steps=MAX_STEPS,
        learning_rate=2e-4, fp16=True, bf16=False,
        logging_steps=10, optim='paged_adamw_32bit',
        weight_decay=0.01, lr_scheduler_type='cosine',
        seed=42, output_dir='/tmp/mironlaw',
        save_strategy='no', report_to='none',
        remove_unused_columns=False,
    ),
)
trainer.train()
print('Egitim tamamlandi!')

# ── Adapter kaydet ────────────────────────────────────────────────────────────
print(f'LoRA adapter kaydediliyor: {ADAPTER_DIR}')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# ── Bellek temizle ve FP16 model yukle → merge → push ─────────────────────────
print('GPU bellegi temizleniyor...')
del trainer
del model
gc.collect()
torch.cuda.empty_cache()
print(f'Bos VRAM: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

print('FP16 base model yukleniyor (merge icin)...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16, device_map='auto', token=HF_TOKEN,
)
peft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
merged = peft_model.merge_and_unload()
print('LoRA merge tamamlandi!')

print(f'HuggingFace push: {HF_REPO} ...')
merged.push_to_hub(HF_REPO, token=HF_TOKEN, max_shard_size='2GB')
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f'TAMAMLANDI: https://huggingface.co/{HF_REPO}')
print(f'MIRONLAW_MODEL={HF_REPO}')